In [163]:
import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)

df = pd.read_csv("100_Unique_QA_Dataset.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

TOKENIZATION OF GIVEN DATASET!

In [164]:
def tokenize(text):
    text = text.lower()
    text = text.replace("?","")
    text = text.replace("'","")
    return text.split()

BUILD VOCABULARY!

In [165]:
vocab = {'<UNK>': 0}

def build_vocab(row):
    tokenised_question = tokenize(row['question'])
    tokenised_answer = tokenize(row['answer'])
    merged_tokens = tokenised_question + tokenised_answer
    for token in merged_tokens:
        if token not in vocab:
            vocab[str(token)] = len(vocab)
    return vocab

# df.apply(build_vocab, axis=1) # or
for index, row in df.iterrows():
    vocab = build_vocab(row)

len(vocab)


324

ENCODING!

In [166]:
import torch

def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

CONVERT TO TENSORS!

In [167]:
def to_tensor(numpy_array):
    tensor = torch.tensor(numpy_array, dtype=torch.long)
    return tensor

DATASET PREP!

In [168]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        # converts to numbers and then to tensors!
        numerical_question = text_to_indices(self.df.iloc[idx]['question'], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[idx]['answer'], self.vocab)
        return to_tensor(numerical_question), to_tensor(numerical_answer)

In [169]:
dataset = QADataset(df, vocab)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

BUILDING THE RNN ARCHITECTURE!

In [170]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)
    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        return output

In [171]:
learning_rate = 0.001
epochs = 20

model = SimpleRNN(len(vocab))

loss_fun = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

TRAINING LOOP!

In [172]:
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    # clear prev grads!
    optimizer.zero_grad()
    # forward pass
    output = model(question)
    # loss -> output shape (1,324) - (1)
    loss = loss_fun(output, answer[0])
    # gradients
    loss.backward()
    # update
    optimizer.step()
    # calculate the total loss!
    total_loss = total_loss + loss.item()
  avg_loss = total_loss/len(dataloader)
  print(f"Epoch: {epoch+1}, Loss: {avg_loss:4f}")

Epoch: 1, Loss: 5.807304
Epoch: 2, Loss: 5.042532
Epoch: 3, Loss: 4.163163
Epoch: 4, Loss: 3.486148
Epoch: 5, Loss: 2.910364
Epoch: 6, Loss: 2.376625
Epoch: 7, Loss: 1.885071
Epoch: 8, Loss: 1.461213
Epoch: 9, Loss: 1.119057
Epoch: 10, Loss: 0.847632
Epoch: 11, Loss: 0.654162
Epoch: 12, Loss: 0.506809
Epoch: 13, Loss: 0.403221
Epoch: 14, Loss: 0.326169
Epoch: 15, Loss: 0.266571
Epoch: 16, Loss: 0.221478
Epoch: 17, Loss: 0.186799
Epoch: 18, Loss: 0.160147
Epoch: 19, Loss: 0.136787
Epoch: 20, Loss: 0.118407


PREDICTION ON THE GIVEN QUESTION!

In [173]:
def predict(model, question, threshold=0.5):
    # to numbers
    numerical_question = text_to_indices(question, vocab)
    # to tensors
    question_tensor = to_tensor(numerical_question).unsqueeze(0)
    # model 
    output = model(question_tensor)
    # getting probs
    probs = torch.nn.functional.softmax(output, dim=1)
    # max values and its index: they are tensors: use .item()
    value, index = torch.max(probs, dim=1)

    if value.item() < threshold:
        print("I don't know")

    print(list(vocab.keys())[index.item()])

In [174]:
predict(model, "What is the largest planet in our solar system?")

jupiter
